Классификация: превышает ли значение CC50 медианное значение выборки

In [10]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# загрузка данных
X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')

# функция для обучения и оценки
def evaluate_classifier(y_train_path, y_test_path, model, model_name, task_name):
    y_train = pd.read_csv(y_train_path).values.ravel()
    y_test = pd.read_csv(y_test_path).values.ravel()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n{task_name} - {model_name}")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"  F1-score: {f1_score(y_test, y_pred):.3f}")
    print(f"  ROC-AUC:  {roc_auc_score(y_test, y_proba):.3f}")

# ============================================
# КЛАССИФИКАЦИЯ CC50
# ============================================
print("="*60)
print("КЛАССИФИКАЦИЯ CC50 (выше/ниже медианы)")
print("="*60)

# Logistic Regression
evaluate_classifier(
    'y_train_cc50_high.csv', 'y_test_cc50_high.csv',
    LogisticRegression(max_iter=1000, random_state=42),
    "LogisticRegression", "CC50"
)

# Random Forest
evaluate_classifier(
    'y_train_cc50_high.csv', 'y_test_cc50_high.csv',
    RandomForestClassifier(n_estimators=100, random_state=42),
    "RandomForest", "CC50"
)

# XGBoost
evaluate_classifier(
    'y_train_cc50_high.csv', 'y_test_cc50_high.csv',
    XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    "XGBoost", "CC50"
)

КЛАССИФИКАЦИЯ CC50 (выше/ниже медианы)

CC50 - LogisticRegression
  Accuracy: 0.512
  F1-score: 0.000
  ROC-AUC:  0.521

CC50 - RandomForest
  Accuracy: 0.769
  F1-score: 0.767
  ROC-AUC:  0.868

CC50 - XGBoost
  Accuracy: 0.748
  F1-score: 0.745
  ROC-AUC:  0.863


Вывод:логистическая регрессия оказалась полностью непригодной для классификации CC50 (F1 = 0.000, AUC = 0.521), что указывает на нелинейный характер зависимости между признаками и токсичностью. Ансамблевые методы показали высокое качество: Random Forest (AUC = 0.868, Accuracy = 76.9%) и XGBoost (AUC = 0.863). Это подтверждает, что для предсказания токсичности необходимы нелинейные модели. Наилучшей моделью для CC50 является Random Forest.